In [30]:
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd / "integration.py").exists():
    project_root = cwd.parent
else:
    project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
print("Project root:", project_root)

Project root: c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch


In [31]:
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage


from langchain_ollama import ChatOllama, OllamaEmbeddings


from data_source.wikipedia import WikipediaSource
from data_source.web_search import WebSearchSource


from src.pipeline.indexing_rag import build_vectorstore
from src.pipeline.retrieval_rag import create_retriever


from src.query_translation.multi_query import (
    create_multi_query_retrieval_chain,
)

from src.Reranking.reranking import CrossEncoderReranker


from src.advanced_indexing.raptor import RaptorIndexer


from src.advanced_RAG.Self_RAG.self_rag import SelfRAG
from src.advanced_RAG.Long_Context.long_context import LongContext


from src.memory.memory import ConversationMemory


from src.evaluation.rag_evaluation import RAGEvaluator

In [32]:
# Load environment variables

load_dotenv()


# Local LLM

llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
    base_url="http://127.0.0.1:11434",
)


# Local embedding model

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    base_url="http://127.0.0.1:11434",
)

In [33]:
# Final generation prompt

generation_prompt = ChatPromptTemplate.from_template("""
You are a careful RAG assistant.

Answer the question using ONLY the provided evidence.

Evidence:
{context}

Question:
{question}

Rules:
- Do not invent facts.
- Use only the provided evidence.
- For current or time-sensitive questions, prefer the newest
  evidence with an explicit date and time.
- If sources conflict, prefer the most recent dated evidence.
- Do not use older information when newer evidence is available.
- If the evidence is insufficient, say that the information
  is not available in the retrieved evidence.
- Answer clearly and concisely.

Answer:
""")

generation_chain = generation_prompt | llm | StrOutputParser()

In [34]:
# Data sources

wikipedia = WikipediaSource(
    top_k=5,
)

web_search = WebSearchSource(
    top_k=5,
)


# Reranker

reranker = CrossEncoderReranker(
    top_k=5,
)


# Conversation memory

conversation_memory = ConversationMemory()


# Evaluator

evaluator = RAGEvaluator(
    model="llama3:latest",
    temperature=0,
)

In [35]:
# Ollama answerability check

answerability_prompt = ChatPromptTemplate.from_template("""
Determine whether you can answer the user's question reliably
using your existing knowledge.

Return ONLY one of these labels:

ANSWERABLE
NOT_ANSWERABLE

Rules:
- Return NOT_ANSWERABLE for current, live, latest, recent,
  today's, yesterday's, tomorrow's, or time-sensitive information.
- Return NOT_ANSWERABLE if you are uncertain.
- Return ANSWERABLE only when you are reasonably confident
  that your existing knowledge is sufficient.

Question:
{question}

Decision:
""")

answerability_chain = answerability_prompt | llm | StrOutputParser()

In [36]:
# User query

query = input("Ask a question: ")


# Ask Ollama first

decision = (
    answerability_chain.invoke(
        {
            "question": query,
        }
    )
    .strip()
    .upper()
)

if "NOT_ANSWERABLE" in decision:
    decision = "NOT_ANSWERABLE"

elif "ANSWERABLE" in decision:
    decision = "ANSWERABLE"

else:
    # Conservative fallback
    decision = "NOT_ANSWERABLE"
print(query)
print("\nOllama decision:", decision)

What is the latest situation of the floods in Nepal, including the death toll, missing people, worst-affected areas, and main causes? Use the most recent reliable sources.

Ollama decision: NOT_ANSWERABLE


In [37]:
# Determine retrieval path

if decision == "ANSWERABLE":

    response = llm.invoke(query)
    answer = response.content

    print("Answered directly by Ollama.")

else:

    print("Ollama could not answer reliably.")
    print("External retrieval required.")

Ollama could not answer reliably.
External retrieval required.


In [38]:
# Detect current / time-sensitive queries


def is_current_query(query: str) -> bool:
    keywords = [
        "today",
        "current",
        "latest",
        "now",
        "live",
        "recent",
        "yesterday",
        "tomorrow",
    ]

    query_lower = query.lower()

    return any(keyword in query_lower for keyword in keywords)


current_query = is_current_query(query)

print("Current query:", current_query)

Current query: True


In [39]:
# External retrieval

documents = []

if decision == "NOT_ANSWERABLE":

    if current_query:

        # Current / time-sensitive information
        documents = web_search.retrieve(query)

        print("Source: Tavily Web Search")

        print(
            "Documents:",
            len(documents),
        )

    else:

        # Stable external information
        wikipedia_documents = wikipedia.retrieve(query)
        web_documents = web_search.retrieve(query)

        documents = wikipedia_documents + web_documents

        print(
            "Wikipedia:",
            len(wikipedia_documents),
        )

        print(
            "Web:",
            len(web_documents),
        )

        print(
            "Total:",
            len(documents),
        )

Source: Tavily Web Search
Documents: 5


In [40]:
# Deduplicate external documents

if decision == "NOT_ANSWERABLE":

    unique_documents = {}

    for document in documents:

        url = document.metadata.get("url")

        if url:
            key = url
        else:
            key = document.metadata.get("title", "") + document.page_content[:200]

        if key not in unique_documents:
            unique_documents[key] = document

    documents = list(unique_documents.values())

    print(
        "Unique documents:",
        len(documents),
    )

Unique documents: 5


In [41]:
for i, document in enumerate(documents, 1):
    print(
        i,
        document.metadata.get("title"),
        document.metadata.get("url"),
    )

1 Nepal flash floods LIVE: Nearly 160 killed, over 750 missing https://www.thehindu.com/news/international/nepal-flash-floods-increasing-death-toll-many-missing-rescuers-search-for-survivors-live-updates-august-27-2026/article71394802.ece
2 Nepal floods live: More than 160 killed, over 800 missing, ... https://www.aljazeera.com/news/liveblog/2026/8/27/nepal-floods-live-more-than-160-killed-over-800-missing-rescue-continues
3 Deadly Flash Flooding Hits Nepal https://www.unicefusa.org/stories/deadly-flash-flooding-hits-nepal
4 Nepal flooding: several dead and hundreds of tourists ... https://www.cnn.com/2026/08/26/world/live-news/nepal-flash-flooding-floods-intl
5 Rescuers search for survivors after devastating Nepal flood https://www.reuters.com/business/environment/rescuers-search-survivors-after-devastating-nepal-flood-2026-08-27


In [42]:
# Reranking
if decision == "NOT_ANSWERABLE":
    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=documents,
    )

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [43]:
from datetime import datetime


def extract_document_datetime(document):
    """Extract document datetime from metadata."""
    metadata = getattr(document, "metadata", {}) or {}

    # Try common metadata keys
    for key in ["date", "datetime", "created_at", "published_at", "timestamp"]:
        value = metadata.get(key)

        if value is not None:
            return value

    return None

In [44]:
# Inspect ranking

if decision == "NOT_ANSWERABLE":

    for i, document in enumerate(
        reranked_documents,
        1,
    ):
        print(f"\n--- Rank {i} ---")

        print(
            "Title:",
            document.metadata.get("title"),
        )

        print(
            "URL:",
            document.metadata.get("url"),
        )

        print(
            "Date:",
            extract_document_datetime(document),
        )

        print(document.page_content[:700])


--- Rank 1 ---
Title: Deadly Flash Flooding Hits Nepal
URL: https://www.unicefusa.org/stories/deadly-flash-flooding-hits-nepal
Date: None
Massive flooding swept through Nepal's north-central Rasuwa district in Bagmati province near the Tibetan border on Aug. 26, 2026, leaving more than 400 people missing and at least 95 dead, according to official reports from Nepal. The Bhotekoshi River flooded around 9 a.m. local time, causing widespread damage to villages, roads and hydropower projects. Chinese state media report three dead and more than 250 missing. [...] ## Utility Menu

Aerial view of flash flooding damage in Nepal on Aug. 26, 2026.

## Breadcrumb

# Deadly Flash Flooding Hits Nepal

Hundreds are missing and at least 95 dead after an earthquake and avalanche triggered catastrophic flash flooding in northern Nepal.

Suppo

--- Rank 2 ---
Title: Nepal floods live: More than 160 killed, over 800 missing, ...
URL: https://www.aljazeera.com/news/liveblog/2026/8/27/nepal-floods-live-m

In [45]:
if decision == "NOT_ANSWERABLE" and documents:
    vectorstore = build_vectorstore(
        documents=documents,
        embedding_model=embeddings,
        batch_size=32,
    )

In [46]:
# Build vector store

vectorstore = None
retriever = None
multi_query_retriever = None

if decision == "NOT_ANSWERABLE":

    vectorstore = build_vectorstore(
        documents=documents,
        embedding_model=embeddings,
        batch_size=32,
    )

    print("Vector store created.")

Vector store created.


In [47]:
# Create retriever

if decision == "NOT_ANSWERABLE":

    retriever = create_retriever(
        vectorstore=vectorstore,
        k=5,
    )

In [48]:
# Multi-Query retrieval

if decision == "NOT_ANSWERABLE":

    multi_query_retriever = create_multi_query_retrieval_chain(
        retriever=retriever,
        llm=llm,
    )

    retrieved_documents = multi_query_retriever.invoke(query)

    print(
        "Retrieved documents:",
        len(retrieved_documents),
    )

Retrieved documents: 22


c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch\src\query_translation\multi_query.py:40: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


In [49]:
# Rerank stable RAG results

if decision == "NOT_ANSWERABLE":

    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=retrieved_documents,
    )

    print(
        "Stable RAG reranked documents:",
        len(reranked_documents),
    )

Stable RAG reranked documents: 5


In [50]:
# RAPTOR

raptor_leaf = []
raptor_clusters = []

if decision == "NOT_ANSWERABLE":

    raptor = RaptorIndexer(
        llm=llm,
        embeddings=embeddings,
        n_clusters=3,
    )

    raptor.build_tree(
        documents=documents,
    )

    raptor_results = raptor.retrieve(
        query=query,
        k=3,
    )

    raptor_leaf = raptor_results.get(
        "leaf",
        [],
    )

    raptor_clusters = raptor_results.get(
        "clusters",
        [],
    )

    print(
        "RAPTOR leaf results:",
        len(raptor_leaf),
    )

    print(
        "RAPTOR cluster results:",
        len(raptor_clusters),
    )

RAPTOR leaf results: 3
RAPTOR cluster results: 3


In [51]:
# Self-RAG

self_rag_answer = ""

if decision == "NOT_ANSWERABLE":

    self_rag = SelfRAG(
        llm=llm,
        retriever=retriever,
    )

    self_rag_answer = self_rag.invoke(
        query,
        max_retries=2,
    )

    print("Self-RAG completed.")


========== SELF-RAG ==========
Retrieval required: YES

========== Attempt 1 ==========
Retrieved documents: 5
Document 1 relevance: YES
Document 2 relevance: YES
Document 3 relevance: YES
Document 4 relevance: YES
Document 5 relevance: YES
Relevant documents: 5

Generated answer:
According to the latest reliable sources, the situation in Nepal is as follows:

* Death toll: At least 165 people have been killed.
* Missing people: 826 people are still missing.
* Worst-affected areas: The Himalayan flash floods have affected several communities along the Nepal-China border.
* Main causes: The flooding was reportedly caused by an avalanche triggered by an earthquake.

These figures and information are based on the most recent reports from Nepal's National Disaster Risk Reduction and Management Authority and other reliable sources.

Answer supported: YES
Answer useful: YES

Self-RAG accepted the answer.
Self-RAG completed.


In [52]:
# Long-context processing

long_context_answer = ""

if decision == "NOT_ANSWERABLE":

    long_context = LongContext(
        model="llama3:latest",
    )

    long_context_result = long_context.run(
        documents=reranked_documents,
        query=query,
        compress=True,
    )

    if isinstance(
        long_context_result,
        dict,
    ):
        long_context_answer = long_context_result.get(
            "answer",
            "",
        )
    else:
        long_context_answer = str(long_context_result)

In [53]:
# Build final context

context = ""

if decision == "NOT_ANSWERABLE":

    reranked_context = "\n\n".join(
        document.page_content for document in reranked_documents
    )

    if current_query:

        # Current/live queries:

        context = reranked_context

    else:

        # Stable external RAG:

        context_parts = [
            reranked_context,
        ]

        if raptor_leaf:
            context_parts.append(
                "RAPTOR Leaf Evidence:\n"
                + "\n\n".join(str(item) for item in raptor_leaf)
            )

        if raptor_clusters:
            context_parts.append(
                "RAPTOR Cluster Evidence:\n"
                + "\n\n".join(str(item) for item in raptor_clusters)
            )

        if self_rag_answer:
            context_parts.append("Self-RAG Evidence:\n" + self_rag_answer)

        if long_context_answer:
            context_parts.append("Long-Context Evidence:\n" + long_context_answer)

        context = "\n\n".join(context_parts)

    print(
        "Final context length:",
        len(context),
    )

Final context length: 5994


In [54]:
# Final generation

if decision == "NOT_ANSWERABLE":

    answer = generation_chain.invoke(
        {
            "context": context,
            "question": query,
        }
    )

In [55]:
# Direct Ollama answer

if decision == "ANSWERABLE":

    response = llm.invoke(query)

    answer = response.content

In [56]:
# Store conversation memory

conversation_memory.add_message(HumanMessage(content=query))

conversation_memory.add_message(AIMessage(content=answer))

In [57]:
# Evaluation

if decision == "ANSWERABLE":

    answer_relevance = evaluator.evaluate_answer_relevance(
        question=query,
        answer=answer,
    )

    evaluation_results = {
        "answer_relevance": answer_relevance,
    }

else:

    context_relevance = evaluator.evaluate_context_relevance(
        question=query,
        context=context,
    )

    faithfulness = evaluator.evaluate_faithfulness(
        context=context,
        answer=answer,
    )

    answer_relevance = evaluator.evaluate_answer_relevance(
        question=query,
        answer=answer,
    )

    evaluation_results = {
        "context_relevance": context_relevance,
        "faithfulness": faithfulness,
        "answer_relevance": answer_relevance,
    }

print("Evaluation:")
for metric, value in evaluation_results.items():
    print(f"{metric}: {value}")

Evaluation:
context_relevance: RELEVANT

The retrieved context contains information that is directly relevant to answering the question. It provides the latest situation of the floods in Nepal, including the death toll, missing people, worst-affected areas, and main causes. The context also mentions the official reports from Nepal, Chinese state media, and a statement from the UNICEF Nepal country office, which provides additional details and context.
faithfulness: FAITHFUL

The answer accurately summarizes the information provided in the context, including the death toll, number of missing people, worst-affected areas, and main causes of the flooding. The answer does not go beyond the information provided in the context, which is consistent with the note that the evidence does not provide information on the current situation or any updates beyond August 26, 2026.
answer_relevance: RELEVANT


In [58]:
# Final output

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)

print(answer)

print("=" * 60)

print(
    "\nDecision:",
    decision,
)

print(
    "Mode:",
    (
        "Direct Ollama"
        if decision == "ANSWERABLE"
        else ("Current Web RAG" if current_query else "Advanced RAG")
    ),
)


FINAL ANSWER
According to the provided evidence, the latest situation of the floods in Nepal is as follows:

* Death toll: At least 95 people are dead, according to official reports from Nepal.
* Missing people: More than 400 people are missing, according to official reports from Nepal. Chinese state media report more than 250 missing.
* Worst-affected areas: The floods occurred in Nepal's north-central Rasuwa district in Bagmati province near the Tibetan border.
* Main causes: The flooding was reportedly caused by an avalanche triggered by an earthquake.

Note: The evidence does not provide information on the current situation or any updates beyond August 26, 2026.

Decision: NOT_ANSWERABLE
Mode: Current Web RAG
